# Temporal Solar-Tornado Training

This notebook converts labeled, time-ordered limb chunks into the three-channel temporal representation used by the supplied checkpoint, then fine-tunes a YOLO11 detector.

The detector is trained on temporal composites rather than an RNN: each sample contains the current frame plus backward and forward motion channels. A sequence is defined by a fixed limb `segment_id` and consecutive timestamps.

## 1. Expected source data

Use standard YOLO detection directories: `images/{train,val}` and `labels/{train,val}`. Each chunk must be named `<YYYY-MM-DDTHHMMSSZ>_<segment_id>.png`; its label has the same stem and normalized rows `class x_center y_center width height`. Missing or empty labels are treated as negative frames.

Split entire events or observing intervals before this notebook. Randomly separating neighbouring frames leaks nearly identical observations into validation.

In [ ]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
if not (ROOT / 'models').exists() and (ROOT.parent / 'models').exists():
    ROOT = ROOT.parent

SOURCE_ROOT = ROOT / 'data' / 'source_dataset'
TEMPORAL_ROOT = ROOT / 'data' / 'temporal_dataset'
DATA_YAML = TEMPORAL_ROOT / 'data.yaml'

# Reproduce from generic YOLO11 weights. To continue fine-tuning the supplied
# detector, use: STARTING_WEIGHTS = ROOT / 'models' / 'temporal_model.pt'
STARTING_WEIGHTS = 'yolo11n.pt'
MAX_GAP_SECONDS = 120
IMAGE_SIZE = 768
EPOCHS = 90
BATCH_SIZE = 8
WORKERS = 4
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RUN_NAME = 'temporal_showcase'
SEED = 42

print(f'root: {ROOT}')
print(f'device: {DEVICE}')

## 2. Validate filenames, images, and labels

The temporal neighbour lookup depends on filenames. This validation fails early on malformed timestamps, image geometry mismatches, unsupported class IDs, or non-normalized boxes.

In [ ]:
def parse_chunk_path(path: Path) -> tuple[datetime, int]:
    parts = path.stem.rsplit('_', 1)
    if len(parts) != 2:
        raise ValueError(f'Expected <timestamp>_<segment_id>: {path.name}')
    timestamp, segment_text = parts
    return datetime.strptime(timestamp, '%Y-%m-%dT%H%M%SZ'), int(segment_text)

def validate_label(path: Path) -> int:
    if not path.exists() or not path.read_text(encoding='utf-8').strip():
        return 0
    count = 0
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f'{path}:{line_number}: expected 5 fields')
        class_id = int(fields[0])
        box = np.asarray(fields[1:], dtype=float)
        if class_id != 0:
            raise ValueError(f'{path}:{line_number}: only class 0 (tornado) is supported')
        if not np.all((0.0 <= box) & (box <= 1.0)):
            raise ValueError(f'{path}:{line_number}: normalized box values must be in [0, 1]')
        if box[2] <= 0 or box[3] <= 0:
            raise ValueError(f'{path}:{line_number}: box width/height must be positive')
        count += 1
    return count

source_summary = {}
split_stems = {}
for split in ('train', 'val'):
    image_dir = SOURCE_ROOT / 'images' / split
    label_dir = SOURCE_ROOT / 'labels' / split
    paths = sorted(image_dir.glob('*.png'))
    if not paths:
        raise FileNotFoundError(f'No PNG chunks found in {image_dir}')
    shapes = set()
    boxes = 0
    segments = set()
    for path in tqdm(paths, desc=f'validating {split}'):
        _, segment_id = parse_chunk_path(path)
        segments.add(segment_id)
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise ValueError(f'OpenCV could not read {path}')
        shapes.add(image.shape)
        boxes += validate_label(label_dir / f'{path.stem}.txt')
    if len(shapes) != 1:
        raise ValueError(f'{split} contains mixed image shapes: {sorted(shapes)}')
    split_stems[split] = {path.stem for path in paths}
    source_summary[split] = {
        'frames': len(paths), 'boxes': boxes, 'segments': len(segments), 'shape': next(iter(shapes))
    }

overlap = split_stems['train'] & split_stems['val']
if overlap:
    examples = sorted(overlap)[:5]
    raise ValueError(f'{len(overlap)} frame stems occur in both train and val, e.g. {examples}')

source_summary

## 3. Convert sequences to temporal images

For each split, frames are grouped by `segment_id` and ordered by timestamp. A neighbour is accepted only when the gap is at most 120 seconds. Sequence boundaries therefore get a zero difference channel. Bounding boxes are copied unchanged because all three channels share the current frame's geometry.

In [ ]:
def read_gray(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'OpenCV could not read {path}')
    return image

def temporal_image(current: np.ndarray, previous: np.ndarray, following: np.ndarray) -> np.ndarray:
    return cv2.merge([
        current, cv2.absdiff(current, previous), cv2.absdiff(following, current)
    ])

def neighbour_map(paths: list[Path]) -> dict[Path, tuple[Path, Path]]:
    sequences = defaultdict(list)
    for path in paths:
        timestamp, segment_id = parse_chunk_path(path)
        sequences[segment_id].append((timestamp, path))
    neighbours = {}
    for sequence in sequences.values():
        sequence.sort(key=lambda item: item[0])
        for index, (timestamp, path) in enumerate(sequence):
            previous = path
            following = path
            if index > 0:
                previous_time, previous_path = sequence[index - 1]
                if (timestamp - previous_time).total_seconds() <= MAX_GAP_SECONDS:
                    previous = previous_path
            if index + 1 < len(sequence):
                following_time, following_path = sequence[index + 1]
                if (following_time - timestamp).total_seconds() <= MAX_GAP_SECONDS:
                    following = following_path
            neighbours[path] = (previous, following)
    return neighbours

def build_temporal_split(split: str) -> dict:
    source_images = SOURCE_ROOT / 'images' / split
    source_labels = SOURCE_ROOT / 'labels' / split
    output_images = TEMPORAL_ROOT / 'images' / split
    output_labels = TEMPORAL_ROOT / 'labels' / split
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)
    # These folders are generated artifacts. Clear matching files so an older
    # dataset build cannot silently contaminate the current training run.
    for stale in output_images.glob('*.png'):
        stale.unlink()
    for stale in output_labels.glob('*.txt'):
        stale.unlink()
    cache_path = TEMPORAL_ROOT / 'labels' / f'{split}.cache'
    if cache_path.exists():
        cache_path.unlink()

    paths = sorted(source_images.glob('*.png'))
    neighbours = neighbour_map(paths)
    nonzero_backward = 0
    nonzero_forward = 0
    for path in tqdm(paths, desc=f'building {split}'):
        previous_path, following_path = neighbours[path]
        current = read_gray(path)
        previous = current if previous_path == path else read_gray(previous_path)
        following = current if following_path == path else read_gray(following_path)
        composite = temporal_image(current, previous, following)
        nonzero_backward += int(np.any(composite[:, :, 1]))
        nonzero_forward += int(np.any(composite[:, :, 2]))
        destination = output_images / path.name
        if not cv2.imwrite(str(destination), composite):
            raise OSError(f'Failed to write {destination}')

        source_label = source_labels / f'{path.stem}.txt'
        destination_label = output_labels / f'{path.stem}.txt'
        if source_label.exists():
            shutil.copy2(source_label, destination_label)
        else:
            destination_label.write_text('', encoding='utf-8')
    return {
        'frames': len(paths), 'backward_motion_frames': nonzero_backward,
        'forward_motion_frames': nonzero_forward,
    }

temporal_summary = {split: build_temporal_split(split) for split in ('train', 'val')}
TEMPORAL_ROOT.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(TEMPORAL_ROOT.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': {0: 'tornado'},
}, sort_keys=False), encoding='utf-8')

print(yaml.safe_dump(temporal_summary, sort_keys=False))
print(DATA_YAML.read_text(encoding='utf-8'))

## 4. Inspect a temporal sample

Motion channels should contain coherent, sparse changes—not unrelated scenes. All-zero motion is expected at sequence boundaries and large gaps.

In [ ]:
train_temporal_paths = sorted((TEMPORAL_ROOT / 'images' / 'train').glob('*.png'))
sample_path = train_temporal_paths[len(train_temporal_paths) // 2]
sample = cv2.imread(str(sample_path), cv2.IMREAD_COLOR)
if sample is None:
    raise ValueError(f'Could not read {sample_path}')

titles = ['current', '|current - previous|', '|next - current|']
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for channel, (ax, title) in enumerate(zip(axes, titles)):
    ax.imshow(sample[:, :, channel], cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
fig.suptitle(sample_path.name)
plt.tight_layout()
plt.show()

## 5. Train YOLO11

Color augmentation is disabled because channel identity carries temporal meaning. Spatial transformations are still applied consistently across the three channels. Training artifacts are written under `runs/detect/temporal_showcase*`; the supplied checkpoint is never overwritten.

In [ ]:
model = YOLO(str(STARTING_WEIGHTS))
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    project=str(ROOT / 'runs' / 'detect'),
    name=RUN_NAME,
    exist_ok=False,
    pretrained=True,
    optimizer='auto',
    patience=20,
    seed=SEED,
    deterministic=True,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    fliplr=0.5,
    flipud=0.0,
    close_mosaic=10,
)

best_checkpoint = Path(model.trainer.best)
last_checkpoint = Path(model.trainer.last)
print(f'best: {best_checkpoint}')
print(f'last: {last_checkpoint}')

## 6. Validate the best checkpoint

Report validation metrics from the event-separated validation set. For a tracking-oriented project, follow this detector evaluation with sequence-level tracking metrics and qualitative track inspection.

In [ ]:
best_model = YOLO(str(best_checkpoint))
validation = best_model.val(
    data=str(DATA_YAML), imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE, split='val'
)

metrics = validation.results_dict if hasattr(validation, 'results_dict') else validation
metrics